# YOLO Classification Fine-tuning Example

## YOLO Classification Fine-tuning

This notebook trains a **new YOLO image classifier** on a custom 4-class person dataset.

```
Dataset (train/ + test/) → YOLO fine-tuning → Custom classifier → Inference
```

### Pipeline

| Step | Description |
|------|-------------|
| 1. Download | Fetch the person-image dataset from GitHub |
| 2. Train | Fine-tune a pretrained YOLO classifier on 4 classes |
| 3. Infer | Run the trained model on new images |

### Dataset — 4 person categories

| Class | Folder |
|-------|--------|
| 0 | `00DzeeShah` |
| 1 | `01RobinHiggins` |
| 2 | `02SayaAkane` |
| 3 | `03YukaKawamura` |

### Base models for fine-tuning

| Model | Params | Speed |
|-------|--------|-------|
| `yolo11x-cls.pt` | 28.4 M | slowest / most accurate |
| `yolo11l-cls.pt` | 12.9 M | — |
| `yolo11m-cls.pt` | 10.4 M | — |
| `yolo11s-cls.pt` |  5.5 M | — |
| `yolo11n-cls.pt` |  1.5 M | fastest / lightest |

Change `--base` when running `--train` to switch the starting model.

📄 [Ultralytics YOLO docs](https://docs.ultralytics.com/)

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/Practical-ML/yolo"
!mkdir -p "{PROJECT_PATH}"
%cd "{PROJECT_PATH}"
!ls

Mounted at /content/drive
/content/drive/MyDrive/Practical-ML/yolo
classification.py  detection.py  images  yolo11n-cls.pt  yolo11n.pt


In [2]:
# Install Ultralytics
!pip install ultralytics -q

import ultralytics
ultralytics.checks()  # check environment

Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 20.8/107.7 GB disk)


In [ ]:
# Download person-image dataset from GitHub
import os

if not os.path.exists(f'{PROJECT_PATH}/images_fine'):
    !git clone --depth=1 --filter=blob:none --sparse https://github.com/mastnk/practical-ml /tmp/practical-ml -q
    !cd /tmp/practical-ml && git sparse-checkout set multiclass/images
    !cp -r /tmp/practical-ml/multiclass/images "{PROJECT_PATH}/images_fine"
    print('Images downloaded.')
else:
    print('Images already exist, skipping.')

%cd "{PROJECT_PATH}"
!ls images/train images/test

## Dataset Structure

The downloaded dataset is organized by class into `train/` and `test/` subfolders:

```
images_fine/
├── train/
│   ├── 00DzeeShah/
│   ├── 01RobinHiggins/
│   ├── 02SayaAkane/
│   └── 03YukaKawamura/
└── test/
    ├── 00DzeeShah/
    ├── 01RobinHiggins/
    ├── 02SayaAkane/
    └── 03YukaKawamura/
```

YOLO expects a `val/` folder instead of `test/`.  
The training script automatically creates a `val → test` symlink if it does not exist.

## Training Settings

Change `--base`, `--epochs`, `--imgsz`, or `--batch` when running `--train`.

| Flag | Default | Description |
|------|---------|-------------|
| `--base` | `yolo11n-cls.pt` | Pretrained model used as the starting point |
| `--epochs` | `50` | Number of training epochs |
| `--imgsz` | `224` | Input image size (pixels) |
| `--batch` | `32` | Batch size |
| `--data` | `images` | Path to the dataset folder |

After training, the best model is saved to `runs/classify/train/weights/best.pt`.

In [ ]:
# Save this cell as a Python file (Execute after editing)
%%writefile classification_fine.py
"""YOLO Classification Fine-tuning — command-line interface.

Training:
  %run classification_fine.py --train [options]

Inference (after training):
  %run classification_fine.py --url  <image_url>  [--disp] [--top_k N]
  %run classification_fine.py --file <image_path> [--disp] [--top_k N]
  %run classification_fine.py --dir  <image_dir>  [--disp] [--top_k N]
"""

import argparse
import glob
import os

from ultralytics import YOLO
from PIL import Image
import requests
from io import BytesIO

# ---- IPython display (works when executed via %run in Colab) -----
try:
    from IPython.display import display as ipy_display
    _has_ipy = True
except ImportError:
    _has_ipy = False

# ---- Configuration -----------------------------------------------
PROJECT_PATH  = '/content/drive/MyDrive/Practical-ML/yolo_fine'
TRAINED_MODEL = 'runs/classify/train/weights/best.pt'

# ---- Display helper ----------------------------------------------
def show(image, label: str, disp: bool) -> None:
    """When --disp is active, display the image then print the filename/URL."""
    if disp:
        if _has_ipy:
            ipy_display(image)
        print(label)

# ---- Training ----------------------------------------------------
def train(data_dir: str, base: str, epochs: int, imgsz: int, batch: int) -> None:
    """Fine-tune a pretrained YOLO classifier on the custom dataset."""
    # YOLO classification expects val/, but the dataset has test/
    val_dir  = os.path.join(data_dir, 'val')
    test_dir = os.path.join(data_dir, 'test')
    if not os.path.exists(val_dir) and os.path.exists(test_dir):
        os.symlink(os.path.abspath(test_dir), val_dir)
        print(f'Created symlink: {val_dir} -> {test_dir}')

    model = YOLO(base)
    model.train(
        data=os.path.abspath(data_dir),
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
    )
    print(f'\nTraining complete. Best model saved to: {TRAINED_MODEL}')

# ---- Inference functions -----------------------------------------
def classify_url(url: str, model, top_k: int = 5, disp: bool = False):
    """Download an image from a URL and classify it with the trained model."""
    image   = Image.open(BytesIO(requests.get(url).content)).convert('RGB')
    results = model.predict(image, verbose=False)
    probs   = results[0].probs
    names   = results[0].names
    pairs   = [(names[idx], float(conf))
               for idx, conf in zip(probs.top5[:top_k], probs.top5conf[:top_k])]
    show(image, url, disp)
    for i, (name, conf) in enumerate(pairs, 1):
        print(f'{i:2d}: {name:<25} {conf*100:.1f}%')
    return pairs


def classify_file(path: str, model, top_k: int = 5, disp: bool = False):
    """Classify a single local image file with the trained model."""
    image   = Image.open(path).convert('RGB')
    results = model.predict(image, verbose=False)
    probs   = results[0].probs
    names   = results[0].names
    pairs   = [(names[idx], float(conf))
               for idx, conf in zip(probs.top5[:top_k], probs.top5conf[:top_k])]
    show(image, path, disp)
    for i, (name, conf) in enumerate(pairs, 1):
        print(f'{i:2d}: {name:<25} {conf*100:.1f}%')
    return pairs


def classify_dir(directory: str, model, top_k: int = 5, disp: bool = False):
    """Classify all images in a directory with the trained model."""
    exts = ['jpg', 'jpeg', 'png', 'bmp', 'JPG', 'JPEG', 'PNG', 'BMP']
    filepaths = []
    for ext in exts:
        filepaths += sorted(glob.glob(os.path.join(directory, f'*.{ext}')))

    if not filepaths:
        print(f'No images found in: {directory}')
        return

    for path in filepaths:
        classify_file(path, model, top_k, disp)
        print()


# ---- Argument parsing --------------------------------------------
parser = argparse.ArgumentParser(description='YOLO Classification Fine-tuning')
group  = parser.add_mutually_exclusive_group(required=True)
group.add_argument('--train', action='store_true', help='Train a new classifier on the dataset')
group.add_argument('--url',   type=str,            help='Image URL to classify (inference)')
group.add_argument('--file',  type=str,            help='Path to a single image file (inference)')
group.add_argument('--dir',   type=str,            help='Directory of images to classify (inference)')

# training options
parser.add_argument('--base',   type=str, default='yolo11n-cls.pt',  help='Base model for training (default: yolo11n-cls.pt)')
parser.add_argument('--data',   type=str, default='images_fine',          help='Dataset directory (default: images_fine)')
parser.add_argument('--epochs', type=int, default=50,                 help='Training epochs (default: 50)')
parser.add_argument('--imgsz',  type=int, default=224,                help='Input image size (default: 224)')
parser.add_argument('--batch',  type=int, default=32,                 help='Batch size (default: 32)')

# inference options
parser.add_argument('--model',  type=str, default=TRAINED_MODEL,      help=f'Model path for inference (default: {TRAINED_MODEL})')
parser.add_argument('--top_k',  type=int, default=5,                  help='Number of top predictions (default: 5)')
parser.add_argument('--disp',   action='store_true',                  help='Display image and filename/URL before results')
args = parser.parse_args()

# ---- Run ---------------------------------------------------------
if args.train:
    train(args.data, args.base, args.epochs, args.imgsz, args.batch)
else:
    model = YOLO(args.model)
    if args.url:
        classify_url(args.url,   model, top_k=args.top_k, disp=args.disp)
    elif args.file:
        classify_file(args.file, model, top_k=args.top_k, disp=args.disp)
    elif args.dir:
        classify_dir(args.dir,   model, top_k=args.top_k, disp=args.disp)


## `classification_fine.py` Usage

After running the `%%writefile` cell above, `classification_fine.py` is saved in `PROJECT_PATH`.
Run it with **`%run`** so that inline image display works in Colab.

### Training

```bash
# Train with default settings (yolo11n-cls.pt, 50 epochs)
%run classification_fine.py --train

# Train with a larger model and more epochs
%run classification_fine.py --train --base yolo11s-cls.pt --epochs 100

# Custom batch size and image size
%run classification_fine.py --train --epochs 50 --imgsz 320 --batch 16
```

The best model is saved to `runs/classify/train/weights/best.pt`.

### Inference (after training)

```bash
# Classify a remote image with the trained model
%run classification_fine.py --url https://example.com/photo.jpg --disp

# Classify a single file, show top-3
%run classification_fine.py --file images/test/00DzeeShah/photo.jpg --top_k 3 --disp

# Classify all test images
%run classification_fine.py --dir images/test/00DzeeShah --disp
```

### All options

| Flag | Default | Description |
|------|---------|-------------|
| **Mode** | | |
| `--train` | — | Fine-tune a new classifier |
| `--url <url>` | — | Classify a remote image |
| `--file <path>` | — | Classify a single local file |
| `--dir <path>` | — | Classify all images in a folder |
| **Training** | | |
| `--base <model>` | `yolo11n-cls.pt` | Pretrained model to start from |
| `--data <path>` | `images` | Dataset folder |
| `--epochs <n>` | `50` | Training epochs |
| `--imgsz <n>` | `224` | Input image size |
| `--batch <n>` | `32` | Batch size |
| **Inference** | | |
| `--model <path>` | `runs/classify/train/weights/best.pt` | Trained model path |
| `--disp` | off | Display image and filename/URL before results |
| `--top_k <n>` | `5` | Number of top predictions to show |

## Execution Methods

Use **`%run`** to execute `classification_fine.py` inside the Colab kernel.

| Step | Command | Description |
|------|---------|-------------|
| ① Train | `%run classification_fine.py --train` | Fine-tune the classifier |
| ② Infer (URL) | `%run classification_fine.py --url <url>` | Classify a remote image |
| ③ Infer (file) | `%run classification_fine.py --file <path>` | Classify a local file |
| ④ Infer (dir) | `%run classification_fine.py --dir <path>` | Classify all images in a folder |

Add `--disp` to display each image and its filename / URL before the results.  
Run **① first**, then ②③④ to test the trained model.

In [ ]:
# Execute: fine-tune a new YOLO classifier on the person dataset
%cd "{PROJECT_PATH}"
%run classification_fine.py --train

In [ ]:
# Execute: classify all images in a test class folder (with display)
%cd "{PROJECT_PATH}"
%run classification_fine.py --disp --dir images_fine/test/00DzeeShah

💾 **Don't forget to save this notebook!**

To keep your work, go to **File → Save a copy in Drive** before closing.

[![Return to GitHub](https://img.shields.io/badge/GitHub-Return-black?logo=github)](https://github.com/mastnk/practical-ml)